In [16]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.utils import to_categorical, Sequence
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input
from tensorflow.keras.models import Model
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd

In [17]:
def load_txt_dataset(directory):
    """
    Scans subfolders 'healthy' and 'sick' within the provided directory and returns lists of file paths and corresponding labels.
    healthy  -> label 0
    sick     -> label 1
    """
    filepaths = []
    labels = []
    for label, class_name in enumerate(['healthy', 'sick']):
        class_dir = os.path.join(directory, class_name)
        for fname in os.listdir(class_dir):
            if fname.lower().endswith('.txt'):
                filepaths.append(os.path.join(class_dir, fname))
                labels.append(label)
    return filepaths, labels
       
class TxtDataSequence(Sequence):
    def __init__(self, filepaths, labels, batch_size=32, target_size=(224,224), shuffle=True, augment=False):
        self.filepaths = filepaths
        self.labels = labels
        self.batch_size = batch_size
        self.target_size = target_size
        self.shuffle = shuffle
        self.augment = augment
        self.indices = np.arange(len(self.filepaths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.filepaths) / self.batch_size))

    def __getitem__(self, index):
        start = index * self.batch_size
        end = min((index+1) * self.batch_size, len(self.filepaths))
        batch_indices = self.indices[start:end]
        X = []
        batch_labels = []
        for i in batch_indices:
            f = self.filepaths[i]
            try:
                # Read the first line to detect delimiter
                with open(f, "r") as file:
                    first_line = file.readline().strip()
                    delimiter = ";" if ";" in first_line else None
                img = np.loadtxt(f, delimiter=delimiter)
            except Exception as e:
                print(f"Error reading {f}: {e}")
                continue
            # Ensure image has a channel dimension
            if img.ndim == 2:
                img = np.expand_dims(img, axis=-1)
            # Convert grayscale to 3-channel if needed
            if img.shape[-1] == 1:
                img = np.concatenate([img, img, img], axis=-1)
            # Resize image to target size
            img_resized = cv2.resize(img, self.target_size)
            img_resized = img_resized.astype('float32')

            # Data augmentation (only for training)
            if self.augment:
                # Random horizontal flip
                if np.random.rand() < 0.5:
                    img_resized = cv2.flip(img_resized, 1)
                # Random rotation between -10 and 10 degrees
                angle = np.random.uniform(-10, 10)
                M = cv2.getRotationMatrix2D((self.target_size[0]//2, self.target_size[1]//2), angle, 1.0)
                img_resized = cv2.warpAffine(img_resized, M, self.target_size)
                # Ensure the output has 3 channels after rotation
                if img_resized.ndim == 2:
                    img_resized = np.expand_dims(img_resized, axis=-1)
                    img_resized = np.concatenate([img_resized, img_resized, img_resized], axis=-1)

            # Preprocess input as required by ResNet50
            img_resized = preprocess_input(img_resized)
            X.append(img_resized)
            batch_labels.append(self.labels[i])
        if not X:
            return np.empty((0, *self.target_size, 3)), np.empty((0, 2))
        X = np.array(X)
        y = to_categorical(np.array(batch_labels), num_classes=2)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


In [18]:
# Define dataset directory and load data
dataset_dir = os.path.join('datasets', 'segmented(txt)')
filepaths, labels = load_txt_dataset(dataset_dir)
total_samples = len(filepaths)
print("Total samples:", total_samples)

# Split dataset: 80% training, 20% test (using stratification)
train_filepaths, test_filepaths, train_labels, test_labels = train_test_split(
    filepaths, labels, test_size=0.2, random_state=42, stratify=labels
)
print("Training samples:", len(train_filepaths))
print("Test samples:", len(test_filepaths))
print("Training class distribution:", Counter(train_labels))


Total samples: 258
Training samples: 206
Test samples: 52
Training class distribution: Counter({0: 135, 1: 71})


In [19]:
batch_size = 32

# Use augmentation only for training data
train_sequence = TxtDataSequence(train_filepaths, train_labels, batch_size=batch_size, target_size=(224,224), shuffle=True, augment=True)
test_sequence  = TxtDataSequence(test_filepaths, test_labels, batch_size=batch_size, target_size=(224,224), shuffle=False, augment=False)
all_sequence   = TxtDataSequence(filepaths, labels, batch_size=batch_size, target_size=(224,224), shuffle=False, augment=False)

steps_per_epoch_train = len(train_sequence)
steps_per_epoch_test  = len(test_sequence)
print("Steps per epoch (train):", steps_per_epoch_train)
print("Steps per epoch (test):", steps_per_epoch_test)


Steps per epoch (train): 7
Steps per epoch (test): 2


In [20]:
# Build the model using ResNet50 as base
input_shape = (224, 224, 3)
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)

# Freeze the base model layers initially
for layer in base_model.layers:
    layer.trainable = False

# Improved classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(2, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=predictions)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,850,242 (90.98 MB)

 Trainable params: 262,530 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [21]:
# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Calculate class weights based on training distribution (optional)
total_train = len(train_labels)
weight_for_0 = total_train / (2 * train_labels.count(0))
weight_for_1 = total_train / (2 * train_labels.count(1))
class_weights = {0: weight_for_0, 1: weight_for_1}
print("Class weights:", class_weights)

# Set up callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

# Train the model
epochs = 10
history = model.fit(
    train_sequence,
    epochs=epochs,
    steps_per_epoch=steps_per_epoch_train,
    class_weight=class_weights,
    validation_data=test_sequence,
    validation_steps=steps_per_epoch_test,
    callbacks=callbacks,
    verbose=1
)

Class weights: {0: 0.762962962962963, 1: 1.4507042253521127}


c:\Users\2004a\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 44s 5s/step - accuracy: 0.5117 - loss: 1.3345 - val_accuracy: 0.7308 - val_loss: 0.4795 - learning_rate: 0.0010
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 20s 3s/step - accuracy: 0.6369 - loss: 0.9575 - val_accuracy: 0.7115 - val_loss: 0.4595 - learning_rate: 0.0010
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 15s 2s/step - accuracy: 0.7003 - loss: 0.7948 - val_accuracy: 0.7115 - val_loss: 0.4769 - learning_rate: 0.0010
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 13s 2s/step - accuracy: 0.6293 - loss: 0.6965 - val_accuracy: 0.7308 - val_loss: 0.4393 - learning_rate: 0.0010
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 20s 3s/step - accuracy: 0.5632 - loss: 0.8823 - val_accuracy: 0.7308 - val_loss: 0.5046 - learning_rate: 0.0010
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 27s 4s/step - accuracy: 0.5917 - loss: 0.7268 - val_accuracy: 0.8462 - val_loss: 0.4521 - learning_rate: 0.0010
Epoch 7/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 28s 4s/step - accuracy: 0.6864 - loss: 0.6510 - val_accuracy: 0.7500 - val_l

In [22]:
# Evaluate initial training performance
eval_loss, eval_accuracy = model.evaluate(test_sequence, steps=steps_per_epoch_test, verbose=1)
print(f"Test Loss: {eval_loss:.4f}")
print(f"Test Accuracy: {eval_accuracy:.4f}")

# %% [code]
# Fine-tuning: unfreeze the last few layers of the base model
for layer in base_model.layers[-20:]:
    layer.trainable = True

# Recompile with a lower learning rate
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
history_finetune = model.fit(
    train_sequence,
    epochs=10,
    steps_per_epoch=steps_per_epoch_train,
    validation_data=test_sequence,
    validation_steps=steps_per_epoch_test,
    callbacks=callbacks,
    verbose=1
)

# Evaluate after fine-tuning
eval_loss, eval_accuracy = model.evaluate(test_sequence, steps=steps_per_epoch_test, verbose=1)
print(f"Post Fine-tuning Test Loss: {eval_loss:.4f}")
print(f"Post Fine-tuning Test Accuracy: {eval_accuracy:.4f}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 928ms/step - accuracy: 0.7372 - loss: 0.4444
Test Loss: 0.4393
Test Accuracy: 0.7308
Epoch 1/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 38s 4s/step - accuracy: 0.6550 - loss: 0.7069 - val_accuracy: 0.7115 - val_loss: 0.4394 - learning_rate: 1.0000e-05
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 16s 2s/step - accuracy: 0.6475 - loss: 0.7369 - val_accuracy: 0.7308 - val_loss: 0.4401 - learning_rate: 1.0000e-05
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 21s 3s/step - accuracy: 0.7170 - loss: 0.5665 - val_accuracy: 0.7308 - val_loss: 0.4398 - learning_rate: 1.0000e-05
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 16s 2s/step - accuracy: 0.7527 - loss: 0.4990 - val_accuracy: 0.7308 - val_loss: 0.4383 - learning_rate: 5.0000e-06
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 31s 4s/step - accuracy: 0.6534 - loss: 0.6476 - val_accuracy: 0.7308 - val_loss: 0.4368 - learning_rate: 5.0000e-06
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 35s 3s/step - accuracy: 0.7974 - loss: 0.4874 - val_accuracy: 0.7115 - val_loss: 0.4345 - l

In [23]:
# Generate table with predicted and actual classifications for all images
predictions = model.predict(all_sequence, verbose=1)
predicted_labels = np.argmax(predictions, axis=1)

df_results = pd.DataFrame({
    'Filename': filepaths,
    'Actual': labels,
    'Predicted': predicted_labels
})
label_mapping = {0: 'healthy', 1: 'sick'}
df_results['Actual_Class'] = df_results['Actual'].map(label_mapping)
df_results['Predicted_Class'] = df_results['Predicted'].map(label_mapping)
print(df_results)

# Save the entire prediction list to a text file
with open("predictions2.txt", "w") as f:
    f.write(df_results.to_string(index=False))
print("Predictions saved to predictions.txt")

9/9 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step
                                              Filename  Actual  Predicted  \
0    datasets\segmented(txt)\healthy\T0001.1.1.S.20...       0          1   
1    datasets\segmented(txt)\healthy\T0002.1.1.S.20...       0          1   
2    datasets\segmented(txt)\healthy\T0004.1.1.S.20...       0          0   
3    datasets\segmented(txt)\healthy\T0005.1.1.S.20...       0          0   
4    datasets\segmented(txt)\healthy\T0006.1.1.S.20...       0          1   
..                                                 ...     ...        ...   
253  datasets\segmented(txt)\sick\T0415.1.1.S.2019-...       1          1   
254  datasets\segmented(txt)\sick\T0418.1.1.S.2020-...       1          1   
255  datasets\segmented(txt)\sick\T0422.1.1.S.2019-...       1          1   
256  datasets\segmented(txt)\sick\T0424.1.1.S.2019-...       1          1   
257  datasets\segmented(txt)\sick\T0425.1.1.S.2020-...       1          1   

    Actual_Class Predicted_Class  
0  

In [15]:
# %% [code]
# Evaluate the model on the test set and compute the confusion matrix
predictions = []
true_labels_arr = []

for X_batch, y_batch in test_sequence:
    if X_batch.size == 0:
        break
    preds = model.predict(X_batch)
    predictions.extend(np.argmax(preds, axis=1))
    true_labels_arr.extend(np.argmax(y_batch, axis=1))

if len(true_labels_arr) == 0:
    raise ValueError("No test samples were processed. Check your data generator.")

predictions = np.array(predictions)[:len(test_filepaths)]
true_labels_arr = np.array(true_labels_arr)[:len(test_filepaths)]

cm = confusion_matrix(true_labels_arr, predictions)
print("Confusion Matrix:\n", cm)

if cm.size == 4:
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    print(f"Sensitivity (TPR): {sensitivity:.4f}")
    print(f"Specificity (TNR): {specificity:.4f}")
else:
    print("Confusion matrix dimensions unexpected for binary classification.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Confusion Matrix:
 [[30  4]
 [ 6 12]]
Sensitivity (TPR): 0.6667
Specificity (TNR): 0.8824
